In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import cv2
from torchvision import models, transforms


# Fix: determinism for stability/reproducibility in Kaggle runs (score-neutral, stability-only)
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)

DATA_ROOT = "/kaggle/input/cassava-leaf-disease-classification"
TRAIN_CSV = os.path.join(DATA_ROOT, "train.csv")
SAMPLE_SUB = os.path.join(DATA_ROOT, "sample_submission.csv")
TRAIN_IMG_DIR = os.path.join(DATA_ROOT, "train_images")
TEST_IMG_DIR = os.path.join(DATA_ROOT, "test_images")

assert os.path.exists(TRAIN_CSV), f"Missing: {TRAIN_CSV}"
assert os.path.exists(SAMPLE_SUB), f"Missing: {SAMPLE_SUB}"
assert os.path.isdir(TRAIN_IMG_DIR), f"Missing: {TRAIN_IMG_DIR}"
assert os.path.isdir(TEST_IMG_DIR), f"Missing: {TEST_IMG_DIR}"

train_df = pd.read_csv(TRAIN_CSV)
sample_df = pd.read_csv(SAMPLE_SUB)

num_classes = int(train_df["label"].nunique())
print("Train rows:", len(train_df), "Classes:", num_classes)
print("Test rows (sample_submission):", len(sample_df))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


Train rows: 18721 Classes: 5
Test rows (sample_submission): 2676


device(type='cuda')

In [2]:
# Fix: remove dependency on unavailable build_transforms; use torchvision transforms instead.
# Note: Cassava baseline typically uses 224-512; 224 keeps runtime within 600s while still reasonable.
IMG_SIZE = 224
train_tfms = transforms.Compose(
    [
        transforms.ToPILImage(),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)

valid_tfms = transforms.Compose(
    [
        transforms.ToPILImage(),
        transforms.Resize(int(IMG_SIZE * 1.14)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)


class CassavaDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_id = self.df.loc[idx, "image_id"]
        path = os.path.join(self.img_dir, image_id)
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Failed to read image: {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform is not None:
            img = self.transform(img)
        if self.is_test:
            return img, image_id
        label = int(self.df.loc[idx, "label"])
        return img, label


# Simple split to train/valid for sanity; training still uses only train data (no leakage).
# (Score improvement vs. no training; does not change evaluation semantics for submission.)
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, valid_idx = next(splitter.split(train_df["image_id"], train_df["label"]))

tr_df = train_df.iloc[train_idx].reset_index(drop=True)
va_df = train_df.iloc[valid_idx].reset_index(drop=True)

tr_ds = CassavaDataset(tr_df, TRAIN_IMG_DIR, transform=train_tfms, is_test=False)
va_ds = CassavaDataset(va_df, TRAIN_IMG_DIR, transform=valid_tfms, is_test=False)

batch_size = 32 if torch.cuda.is_available() else 16
num_workers = 2

tr_loader = DataLoader(
    tr_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)
va_loader = DataLoader(
    va_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

len(tr_loader), len(va_loader)


(526, 59)

In [3]:
# Fix: missing external "model.net/build_model" -> use standard torchvision resnet50 classifier.
# This preserves the "ResNet50 classification" core idea and produces a working model.
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=2)


def accuracy_from_logits(logits, y):
    pred = torch.argmax(logits, dim=1)
    return (pred == y).float().mean().item()


# Train a small number of epochs to keep runtime bounded and produce non-random submission.
# (Necessary since provided weights path/config are unavailable in this environment.)
epochs = 2

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
for epoch in range(1, epochs + 1):
    model.train()
    tr_loss = 0.0
    tr_acc = 0.0
    for x, y in tr_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        tr_loss += loss.item()
        tr_acc += accuracy_from_logits(logits.detach(), y)

    scheduler.step()

    model.eval()
    va_loss = 0.0
    va_acc = 0.0
    with torch.no_grad():
        for x, y in va_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(x)
                loss = criterion(logits, y)
            va_loss += loss.item()
            va_acc += accuracy_from_logits(logits, y)

    tr_loss /= max(1, len(tr_loader))
    tr_acc /= max(1, len(tr_loader))
    va_loss /= max(1, len(va_loader))
    va_acc /= max(1, len(va_loader))

    print(
        f"Epoch {epoch}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | valid loss {va_loss:.4f} acc {va_acc:.4f}"
    )


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 103MB/s]
/tmp/ipykernel_55/1638421831.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_55/1638421831.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_55/1638421831.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Epoch 1/2 | train loss 0.5954 acc 0.7882 | valid loss 0.5070 acc 0.8270
Epoch 2/2 | train loss 0.4194 acc 0.8544 | valid loss 0.3656 acc 0.8794


In [4]:
# Fix: create test loader based on sample_submission to guarantee correct length/order.
test_ds = CassavaDataset(
    sample_df[["image_id"]].copy(), TEST_IMG_DIR, transform=valid_tfms, is_test=True
)
test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

# Inference
model.eval()
all_ids = []
all_preds = []

with torch.no_grad():
    for x, image_ids in test_loader:
        x = x.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(x)
        preds = torch.argmax(logits, dim=1).detach().cpu().numpy().astype(int).tolist()
        all_preds.extend(preds)
        all_ids.extend(list(image_ids))

# Sanity checks for Kaggle submission validity
assert len(all_ids) == len(
    sample_df
), f"Prediction length mismatch: {len(all_ids)} vs {len(sample_df)}"
assert (
    all_ids == sample_df["image_id"].tolist()
), "Order mismatch vs sample_submission; must match exactly."

sub = pd.DataFrame({"image_id": all_ids, "label": all_preds})
sub_path = "./submission.csv"
sub.to_csv(sub_path, index=False)

print("Wrote:", sub_path)
print(sub.head())
print("Rows:", len(sub))


/tmp/ipykernel_55/1902900592.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Wrote: ./submission.csv
         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      1
3  1234555380.jpg      1
4  1234571117.jpg      3
Rows: 2676


In [5]:
# Final validation of submission format/length
chk = pd.read_csv("./submission.csv")
print(chk.shape)
print(chk.columns.tolist())
print(chk.dtypes)
print(chk.head())
assert (
    chk.shape[0] == pd.read_csv(SAMPLE_SUB).shape[0]
), "Invalid submission: wrong number of rows"
assert chk.columns.tolist() == [
    "image_id",
    "label",
], "Invalid submission: wrong columns"
assert (
    chk["label"].between(0, num_classes - 1).all()
), "Invalid submission: labels out of range"

(2676, 2)
['image_id', 'label']
image_id    object
label        int64
dtype: object
         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      1
3  1234555380.jpg      1
4  1234571117.jpg      3
